# Queries

On this page, we will go over all the queries available with a TopicDatabase.

First, let's load a TopicDatabase from a file as well as our embedding model. This database contains data from the 20 Newsgroups dataset - you can check out the [20-Newsgroups example](/newsgroups.html) to see how it was made.

In [ ]:
import thematic_search as ts
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

topicdb = ts.TopicDatabase.from_file("docs/source/20ng-topicdb.tm.zip")
topicdb.embedding_model = model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FileNotFoundError: [Errno 2] No such file or directory: 'thematic-search\\docs\\source\\20ng-topicdb.tm.zip'

## Sample v.s. Topic Queries

The data in a TopicDatabase consists of two distinct types of object. There are *topics*, and there are *samples*. Note that by *samples* we mean both the original documents and their embeddings, and we identify both with their index. 

For this reason, there are two query classes in Thematic Search; ``SampleQuery`` and ``TopicQuery``. These classes correspond to the rows and columns of the cluster inclusion strength matrix respectively. When each class is initialized, it has a ``self.indices`` property that holds the input to the query -- a list of topics indices for TopicQuery and a list of sample indices for SampleQuery.

Each class has a method corresponding to every arrow coming out of it in the diagram. For example, we see that $D_I$ has an outbound arrow "documents" that sends it to the dataframe consisting of document metadata. This corresponds to the method ``SampleQuery.documents()`` which outputs the rows of the document metadata dataframe corresponding to the indices in `SampleQuery.value`.

Here's that example in code:

In [ ]:
query = ts.SampleQuery(topicdb, [0,5,120,55])
result = query.metadata()

result

,post,newsgroup
0,\n\nI am sure some bashers of Pens fans are pr...,rec.sport.hockey
5,\n\nBack in high school I worked as a lab assi...,sci.electronics
120,I am working with Visual Basic v2.0 for window...,comp.os.ms-windows.misc
55,"\n My rule of thumb is ""Don't give rides to ...",rec.motorcycles


## Query Entrypoints

Instead of initializing a new `SampleQuery` or `TopicQuery`, a TopicDatabase comes with a query entrypoint `TopicDatabase.q`. This initializes a `RootQuery` object that can be queried using any of the following methods: 
- `docs_where("query_string")` - retuns an SampleQuery with indices of documents satisfying the query string (using Pandas' .query() method).
- `topics_where("query_string")` - returns a TopicQuery with topics whose metadata satisfies the query string (using Pandas' query() method).
- `from_docs([i1,i2,...,ik])` - returns an SampleQuery with indices `[i1,...,ik]`
- `topic((layer, cluster))` - returns a TopicQuery with topic `(layer, cluster)`
- `topic_name("my-topic-string")` - returns a TopicQuery with the topic whose name is "my-topic-string"
- `topic_idx(15)` - returns a TopicQuery with the topic whose index in `topic_df` is 15
- `neighbours("my search string")` - embeds the string "my search string" using `TopicDatabase.embedding_model` and then returns a SampleQuery with its nearest neighbours.

Let's see these all in code:


In [ ]:
print(
    "docs_where:", topicdb.q.docs_where("newsgroup == 'rec.sport.hockey'")
)
print(
    "topics_where:", topicdb.q.topics_where("layer==0 & cluster<=2")
)
print(
    "from_docs:", topicdb.q.from_docs([10,20,30,40])
)
print(
    "topic:", topicdb.q.topic(1,0)
)
print(
    "topic_name:", topicdb.q.topic_name("alt.atheism")
)
print(
    "topic_idx:", topicdb.q.topic_idx(15)
)
print(
    "neighbours:", topicdb.q.neighbours("comparison of Mac and PC computers")
)

docs_where: SampleQuery(971 samples)
topics_where: TopicQuery(3 topics)
from_docs: SampleQuery(4 samples)
topic: TopicQuery(1 topics)
topic_name: TopicQuery(1 topics)
topic_idx: TopicQuery(1 topics)
neighbours: SampleQuery(15 samples)


In [ ]:
topicdb.topic_df.head()

,name,layer,cluster,uid
index,,,,
0,root,5,0,ABQB
1,alt,1,0,AAQB
2,alt.atheism,0,0,AAAB
3,comp,4,0,ABAB
4,comp.graphics,0,1,AAAC


## Chaining Queries

Many queries output another query object. For example, `SampleQuery.theme()` outputs a `TopicQuery`. This allows us to chain queries together, for example:

In [ ]:
topicdb.q.neighbours("advancements in computer graphics").theme().info()

,name,layer,cluster,uid
index,,,,
1,alt,1,0,AAQB


The diagram is very helpful in this regard, as we can chain queries whenever we can compose two arrows in the diagram. 

## Inside, Topic, and Theme

The last three arrows in the diagram we haven't discussed are `inside`, `topic` and `theme`. These arrows map between TopicQuery and IndexQuery.

 First, `inside` and `topic`; these queries are inverse of each other. `TopicQuery.inside(a)` returns the indices of all documents inside any of the input topics with inclusion strength at least `a`. Conversely, `IndexQuery.topics(a)` returns the set of topics containing at least one of the input indices with inclusion strength at least `a`.

 Finally, `IndexQuery.theme` is the namesake query of this entire package. It returns the topic that can be considered the "theme" of the input documents. For more details on what this means precisely, see [What is Thematic Search?](/theme-query.html)

### Logical Operators

The query functions `inside` and `theme` have keyword argument `logic` that takes either `AND` or `OR` - by default, `OR`. For each query function, setting `logic="AND"` swaps the logical quantifiers as appropriate.

For example `TopicQuery.inside(a, logic="OR")` returns the indices of all documents inside *any* of the input topics with inclusion strength at least `a`. However, `TopicQuery.inside(a, logic="AND")` returns the indices of all documents in *ALL* of the of the input topics with inclusion strength at least `a`.

Similarly, `IndexQuery.topics(a)` returns the set of topics containing *at least one* of the input indices with inclusion strength at least `a`. Whereas `IndexQuery.topics(a)` returns the set of topics containing *ALL* of the input indices with inclusion strength at least `a`.